# Домашняя работа 1 — доли исходов HMDA

## Подготовка

In [1]:
from pathlib import Path
import sys
import pandas as pd

print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("Рабочая папка:", Path.cwd())
DATA = Path("../data")
assert DATA.is_dir(), "Рядом с папкой ноутбука должна находиться data/"

Python: C:\Users\Артёмка\анализ данных\data-analysis-course\.venv\Scripts\python.exe
pandas: 2.2.2
Рабочая папка: C:\Users\Артёмка\анализ данных\data-analysis-course\lessons\01-introduction\work


## Основной набор decisions.csv

In [2]:
home_decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
home_outcomes = home_decisions["action_taken"]
home_n = len(home_decisions)
assert home_n > 0
assert home_outcomes.isna().sum() == 0
assert home_outcomes.isin([1, 2, 3]).sum() == home_n
print(home_decisions.shape)
home_outcomes.value_counts(dropna=False)

(6830, 7)


action_taken
1    5956
3     691
2     183
Name: count, dtype: int64

## H1. Доля отказов

In [3]:
home_denied = home_outcomes.eq(3).sum()
home_denial_rate = home_denied / home_n
print(f'Условие: Найти количество записией со значением 3 и найти их долю')
print(f'Числитель : {home_denied}')
print(f'Знаменатель : {home_n}')
print(f'Процент: {home_denial_rate * 100}')

home_approved = home_outcomes.isin([1, 2]).sum()
assert home_denied + home_approved == home_n
assert 0 <= home_denial_rate <= 1

Условие: Найти количество записией со значением 3 и найти их долю
Числитель : 691
Знаменатель : 6830
Процент: 10.117130307467056


## H2. Одобрение без принятия предложения

In [4]:
home_not_accepted = home_outcomes.eq(2).sum()
home_not_accepted_rate = home_not_accepted / home_n
print(f'Счётчики : {home_not_accepted}')
print(f'Процент : {home_not_accepted_rate*100}')
home_issued = home_outcomes.eq(1).sum()
assert  home_approved - home_issued == home_not_accepted
approved_rate = home_approved / home_n
issued_rate = home_issued / home_n
diff_pp = (approved_rate - issued_rate) * 100
print(f"Доля одобрений: {approved_rate} = {approved_rate * 100} ")
print(f"Доля выдач: {issued_rate} = {issued_rate * 100}")
print(f"Разность: {diff_pp}")

Счётчики : 183
Процент : 2.679355783308931
Доля одобрений: 0.8988286969253294 = 89.88286969253294 
Доля выдач: 0.8720351390922401 = 87.203513909224
Разность: 2.6793557833089343


### Отчёт H2 
Определение. home_not_accepted - число записей с кодом 2 (action_taken == 2), доля home_not_accepted_rate = home_not_accepted / home_n посчитана среди тех же home_n = 6830 записей, что и в H1. Отдельно: home_issued — число выдач (код 1).

Проверка по количествам. assert home_approved - home_issued == home_not_accepted- выполнена на исходных числах, без процентов, чтобы округление не влияло.

Связь разности долей с исходом 2. Одобрения (коды 1 и 2) минус выдачи (код 1) дают ровно код 2. Так как знаменатель home_n общий:

по количествам: home_approved - home_issued = home_not_accepted;

по долям: approved_rate − issued_rate = home_not_accepted / home_n, то есть разность в процентных пунктах равна проценту исхода 2.

Вывод. Разность долей одобрений и выдач - это и есть доля одобрений без принятия предложения (код 2): такие записи входят в одобрения, но не в выдачи.

## H3. Объяснение инструментов

Series и DataFrame. Series - это ячеёки таблицы. DataFrame - таблица из нескольких колонок . В работе home_outcomes = home_decisions["action_taken"] - это Series (один столбец исходов), а home_decisions = pd.read_csv(...) - DataFrame (вся таблица). Поэтому home_outcomes.eq(3).sum() считает по одному столбцу, а home_decisions.shape описывает всю таблицу.

nrows и head. nrows - параметр read_csv, который ограничивает число прочитанных из файла строк ещё на этапе загрузки. head() - метод DataFrame/Series, который показывает первые строки уже загруженных данных, не меняя саму таблицу. В работе home_decisions читался целиком, а для просмотра использовался home_outcomes.value_counts(); head() показал бы первые записи, тогда как pd.read_csv(path, nrows=100) загрузил бы только 100 строк - это разные вещи: одно влияет на данные, другое только на просмотр.

Метод и атрибут. Метод - это действие, вызывается со скобками: home_outcomes.eq(3).sum(). Атрибут - это свойство объекта, читается без скобок: home_decisions.shape. В работе home_outcomes.eq(3) — метод (сравнение), а home_decisions.shape - атрибут (размер таблицы). Ошибка «забыть скобки» (sum вместо sum()) как раз превращает вызов метода в ссылку на него.

Документация. Использован метод pandas.Series.isin - ссылка: https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html. isin принимает список значений и возвращает Series из True/False, причём проверка идёт по каждому элементу. Это подтвердило, что home_outcomes.isin([1, 2]).sum() корректно считает одобрения по двум кодам сразу, в отличие от eq, который сравнивает с одним значением.

## Дополнительная часть

## Исходный снимок для H+1

In [5]:
raw = pd.read_csv(
    DATA / "hmda_2023_dc_raw.csv",
    dtype={"lei": "string", "county_code": "string"},
)
raw_outcomes = raw["action_taken"]
raw_n = len(raw)
assert raw_n > 0
print(raw.shape)
raw_outcomes.value_counts(dropna=False)

(8575, 99)


action_taken
1    5956
4    1585
3     691
2     183
5     160
Name: count, dtype: int64

## H+1. Доли выдач в двух наборах

## Текстовые коды для H+2

In [6]:
text_example = pd.read_csv(
    DATA / "reading_example.csv", dtype={"action_taken": "string"}
)
text_example["action_taken"]

0    1
1    1
2    2
3    2
4    3
5    3
Name: action_taken, dtype: string

## H+2. Сравнение текста с числом и текстом